# S3.1 · 实体对齐（PSI）质量与下游影响

本步**不实现**真实密码学 PSI，只仿真其可观测行为，回答一个工程问题：**对齐做得不好，会损失多少模型价值？**

两类失败模式性质完全不同：
- **漏配**：本该匹配上的人没匹配上 → 样本量减少
- **错配**：把甲的特征接到乙身上 → 注入无关噪声，且属于错误的个人信息关联

In [1]:
ROUND_DP = 4          # 表格展示精度（不影响任何计算结果）
import sys, subprocess, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "registry").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, yaml
CONFIG_PATH = ROOT / "modules/m2_synthetic/configs/scenarios.yaml"
config = yaml.safe_load(open(CONFIG_PATH, encoding="utf-8"))
seed = config.get("seeds", [config.get("seed")])[0]
git = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, cwd=ROOT).stdout.strip()
print("config:", CONFIG_PATH.relative_to(ROOT))
print("seed  :", seed, "| 全部种子:", config.get("seeds"))
print("git   :", git or "(未提交)")
print("numpy :", np.__version__, "| pandas:", pd.__version__)

config: modules/m2_synthetic/configs/scenarios.yaml
seed  : 11 | 全部种子: [11, 22, 33, 44, 55]
git   : 7f29b9e
numpy : 2.3.5 | pandas: 2.3.3


In [2]:
from dataclasses import replace
from modules.m2_synthetic.components.scm_generator import load_scenarios, generate, usable_mask
from modules.m3_alignment.components.psi import simulate_psi, psi_cost, apply_misattribution
from modules.m5_modeling.components import models as M
from sklearn.metrics import roc_auc_score
base = [s for s in load_scenarios(config) if s.name == 'S1_基准'][0]
pd.read_csv(ROOT / 'modules/m3_alignment/results/psi_quality.csv').groupby('match_error_rate')[['n_true_overlap','n_matched','false_negative','recall']].mean().round(ROUND_DP)

,n_true_overlap,n_matched,false_negative,recall
match_error_rate,,,,
0.00,12004.8,12004.8,0.0,1.0000
0.01,12004.8,11873.8,131.0,0.9891
0.03,12004.8,11628.2,376.6,0.9686
0.05,12004.8,11391.6,613.2,0.9489
0.10,12004.8,10789.8,1215.0,0.8988


## 错配对 VFL 增益的影响

In [3]:
pd.read_csv(ROOT / 'modules/m3_alignment/results/misattribution_impact.csv').groupby('misattribution_rate')[['L0','L3a','增益']].mean().round(ROUND_DP)

,L0,L3a,增益
misattribution_rate,,,
0.00,0.7089,0.7868,0.0780
0.01,0.7089,0.7837,0.0748
0.03,0.7089,0.7872,0.0783
0.05,0.7089,0.7809,0.0721
0.10,0.7089,0.7732,0.0643
0.20,0.7089,0.7567,0.0479
0.40,0.7089,0.7376,0.0288


## ECDH-PSI 的成本量级

只用于判断可行性，不是性能基准。

In [4]:
pd.read_csv(ROOT / 'modules/m3_alignment/results/psi_cost.csv').round(2)

,n_a,n_b,comm_bytes,comm_mb,exponentiations,rounds
0,50000,50000,6400000,6.10,200000,2
1,500000,500000,64000000,61.04,2000000,2
2,5000000,5000000,640000000,610.35,20000000,2
3,50000000,10000000,3840000000,3662.11,120000000,2


**结论**：漏配 10% 只损失约 12% 增益，错配 40% 才使增益折半。**实体对齐质量不是本项目的主要风险**——主要风险在互补性与合规折损。